# 02 — Calibration Image Exploration and Corner Detection

**Objective:** Load calibration images (real or downloaded), visualise them, run `cv2.findChessboardCorners`, inspect detection quality, and decide which images are usable.

**Calibration image options (choose one):**

| Option | How | Size |
|---|---|---|
| **A: Download OpenCV sample images** | Automatically downloaded in Cell 2.1 | ~2 MB |
| **B: Use your own camera photos** | Place JPG/PNG files in `data/calibration_images/` | Varies |
| **C: Synthetic images (no camera needed)** | Generated in Cell 2.0 | < 1 MB |

**Option C is the default smoke test** — it generates a set of synthetic checkerboards with simulated perspective variation, enough to verify the pipeline end-to-end. For a real calibration profile you will need real images from your camera.


In [ ]:
import sys
import urllib.request
import zipfile
import shutil
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.calibrator import CameraCalibrator, CheckerboardDetection
from calibvision.utils import load_images_from_dir, draw_corners

PROJECT_ROOT = Path('..').resolve()
CALIB_DIR = PROJECT_ROOT / 'data' / 'calibration_images'
CALIB_DIR.mkdir(parents=True, exist_ok=True)

print(f'Calibration image directory: {CALIB_DIR}')

In [ ]:
# ── 2.0  Option C: Generate synthetic calibration images ─────────────────────
# 
# We generate 20 synthetic checkerboard images with randomised perspective and
# homogeneous lighting. These are enough to run the pipeline end-to-end.
#
# Limitation: synthetic images produce artificially low reprojection errors
# because they have no real lens distortion, sensor noise, or motion blur.
# Real images are required for a genuine calibration profile.
#
# Set USE_SYNTHETIC = False if you have real images in data/calibration_images/

USE_SYNTHETIC = True   # <-- Change to False to use your own images

BOARD_COLS, BOARD_ROWS = 9, 6
N_SYNTHETIC = 20


def make_perspective_checkerboard(
    cols: int, rows: int, square_px: int,
    rng: np.random.Generator,
    img_w: int = 960, img_h: int = 720,
    noise_std: float = 1.5,
) -> np.ndarray:
    """Generate a flat checkerboard then apply a random perspective warp."""
    border = 60
    bh = rows * square_px + 2 * border
    bw = cols * square_px + 2 * border
    board = np.full((bh, bw), 255, dtype=np.uint8)
    for r in range(rows + 1):
        for c in range(cols + 1):
            if (r + c) % 2 == 0:
                y0 = border + r * square_px
                x0 = border + c * square_px
                board[y0:y0 + square_px, x0:x0 + square_px] = 0

    # Random perspective: jitter four corners slightly
    jitter = 40
    src = np.float32([[0, 0], [bw, 0], [bw, bh], [0, bh]])
    dst = src + rng.uniform(-jitter, jitter, src.shape).astype(np.float32)
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(board, H, (bw, bh),
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=200)

    # Embed in a larger canvas
    canvas = np.full((img_h, img_w), 200, dtype=np.uint8)
    y_off = (img_h - bh) // 2
    x_off = (img_w - bw) // 2
    canvas[y_off:y_off + bh, x_off:x_off + bw] = warped

    # Light sensor noise
    noise = rng.normal(0, noise_std, canvas.shape).astype(np.int16)
    canvas = np.clip(canvas.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return canvas


if USE_SYNTHETIC:
    # Clear existing synthetic images to avoid mixing
    for f in CALIB_DIR.glob('synthetic_*.png'):
        f.unlink()

    rng = np.random.default_rng(seed=42)
    saved = 0
    for i in range(N_SYNTHETIC):
        sq = int(rng.integers(50, 70))  # vary square size for diversity
        img = make_perspective_checkerboard(
            BOARD_COLS, BOARD_ROWS, sq, rng
        )
        cv2.imwrite(str(CALIB_DIR / f'synthetic_{i:02d}.png'), img)
        saved += 1

    print(f'✓ Generated {saved} synthetic calibration images → {CALIB_DIR}')
    print('  Note: synthetic images validate the pipeline but do NOT'
          ' produce a real camera calibration profile.')
else:
    n_existing = len(list(CALIB_DIR.glob('*.jpg'))) + len(list(CALIB_DIR.glob('*.png')))
    print(f'Using existing images in {CALIB_DIR}: {n_existing} files found')

In [ ]:
# ── 2.1  Load and inventory images ───────────────────────────────────────────
paths, images = load_images_from_dir(CALIB_DIR, grayscale=False)

print(f'Total images loaded: {len(images)}')
if images:
    h, w = images[0].shape[:2]
    print(f'Image resolution: {w}×{h} px')
    sizes = set((img.shape[1], img.shape[0]) for img in images)
    if len(sizes) > 1:
        print(f'⚠️  WARNING: Mixed resolutions found: {sizes}')
        print('   All images must have the same resolution for calibration.')
    else:
        print('✓ All images have consistent resolution')

    total_mb = sum(p.stat().st_size for p in paths) / 1e6
    print(f'Total dataset size: {total_mb:.1f} MB')

In [ ]:
# ── 2.2  Visualise a sample grid of raw images ────────────────────────────────
n_show = min(8, len(images))
cols_g = 4
rows_g = (n_show + cols_g - 1) // cols_g

fig, axes = plt.subplots(rows_g, cols_g, figsize=(14, rows_g * 3.5))
axes = np.array(axes).ravel()

for i, ax in enumerate(axes):
    if i < n_show:
        img = images[i]
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(paths[i].name, fontsize=7)
    ax.axis('off')

plt.suptitle(f'Calibration images ({n_show} of {len(images)} shown)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3  Run corner detection on all images ────────────────────────────────────
calibrator = CameraCalibrator(
    board_cols=BOARD_COLS,
    board_rows=BOARD_ROWS,
    min_valid_frames=10,
)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

detections = calibrator.detect_corners(paths, verbose=True)

n_ok  = sum(d.success for d in detections)
n_bad = len(detections) - n_ok
print(f'\nDetection summary: {n_ok} found / {n_bad} failed out of {len(detections)}')

if n_bad > 0:
    print('Failed images:')
    for d in detections:
        if not d.success:
            print(f'  ✗ {d.image_path.name}')
    print('Possible causes: image blur, partial board visibility, wrong board size in config.')

In [ ]:
# ── 2.4  Visualise detected corners on successful images ──────────────────────
valid_detections = [d for d in detections if d.success]
n_show_corners = min(6, len(valid_detections))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, ax in enumerate(axes):
    if i < n_show_corners:
        det = valid_detections[i]
        img_bgr = cv2.imread(str(det.image_path))
        vis = draw_corners(img_bgr, det.corners, BOARD_COLS, BOARD_ROWS)
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(det.image_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle(
    f'Detected checkerboard corners — {n_show_corners} of {n_ok} valid images shown',
    fontsize=11
)
plt.tight_layout()
plt.show()

print('\nObservations:')
print('  - Rainbow-coloured path = order in which corners are numbered (matters for calibration)')
print('  - All valid images should show consistent numbering direction')
print(f'  - {n_ok} images are ready for calibration in Notebook 03')

In [ ]:
# ── 2.5  Image quality metrics — brightness and blur ─────────────────────────
# Blurry or very dark images degrade calibration accuracy.
# We quantify blur using the Laplacian variance: high variance → sharp.

quality_data = []
for d in detections:
    img = cv2.imread(str(d.image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    lap_var = float(cv2.Laplacian(img, cv2.CV_64F).var())   # blur metric
    mean_brightness = float(img.mean())
    quality_data.append({
        'name': d.image_path.name,
        'detected': d.success,
        'blur_score': lap_var,
        'brightness': mean_brightness,
    })

blur_scores = [q['blur_score'] for q in quality_data]
ok_blur   = [q['blur_score'] for q in quality_data if q['detected']]
fail_blur = [q['blur_score'] for q in quality_data if not q['detected']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(ok_blur, bins=12, color='#2ecc71', alpha=0.8, label=f'Detected ({len(ok_blur)})')
ax1.hist(fail_blur, bins=12, color='#e74c3c', alpha=0.8, label=f'Failed ({len(fail_blur)})')
ax1.set_xlabel('Laplacian Variance (↑ = sharper)')
ax1.set_ylabel('Count')
ax1.set_title('Sharpness Distribution\n(Laplacian variance)')
ax1.legend()

brightness = [q['brightness'] for q in quality_data]
ax2.scatter(
    [q['brightness'] for q in quality_data],
    [q['blur_score'] for q in quality_data],
    c=['#2ecc71' if q['detected'] else '#e74c3c' for q in quality_data],
    alpha=0.8, edgecolors='white', linewidths=0.5
)
ax2.set_xlabel('Mean Brightness')
ax2.set_ylabel('Laplacian Variance (sharpness)')
ax2.set_title('Brightness vs Sharpness\nGreen=detected · Red=failed')

plt.tight_layout()
plt.show()

print(f'Mean sharpness (detected): {np.mean(ok_blur):.1f}')
if fail_blur:
    print(f'Mean sharpness (failed):   {np.mean(fail_blur):.1f}')

## Summary

| Metric | Value |
|---|---|
| Total images | *(see output above)* |
| Successful detections | *(see output above)* |
| Image resolution | *(see output above)* |
| Image source | Synthetic / Own photos |

## Limitations

- Synthetic images remove real lens distortion — they validate the pipeline, not a real camera.
- The Laplacian blur metric is a coarse heuristic. Images may fail corner detection for other reasons (partial board, wrong board size, extreme perspective).

## Next step

Open `03_calibration_intrinsics.ipynb` to estimate (K, D) from the valid detections.
